# Curriculum Training: Pacing → Racing

Two-phase curriculum:
1. **Solo time trial** — one horse, no opponents. Heavy step penalty forces the agent to learn that pushing = faster = better, but exhaustion = crawling = worse. Teaches pacing.
2. **Race** — transfer the pacing policy, add scripted opponents. Agent learns competitive placement on top of pacing fundamentals.

**Why curriculum?** The cruise controller auto-drives at 13 m/s with zero input. Against weak opponents, doing nothing wins 1st place. PPO never discovers that pushing is beneficial because the per-tick signal is too small. The solo env uses a 20x progress scale + heavy step penalty so each tick of pushing is clearly better than cruising.

**Auto-resume:** Checkpoints save to Google Drive per phase.

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

VERSION = "v5.0"

TRACK = "tracks/test_oval.json"
HORSE_COUNT = 4           # for phase 2 (race)
MAX_STEPS = 5000
N_ENVS = 4

# Phase 1: Solo pacing
SOLO_TIMESTEPS = 500_000

# Phase 2: Racing (fine-tune on race env)
RACE_TIMESTEPS = 500_000
RACE_LR = 1e-4            # lower LR for fine-tuning

# Google Drive paths
DRIVE_BASE = "/content/drive/MyDrive/hr-checkpoints"
SOLO_DIR = f"{DRIVE_BASE}/{VERSION}-solo"
RACE_DIR = f"{DRIVE_BASE}/{VERSION}-race"

RESUME = 'auto'
LOG_EVERY = 10
SAVE_EVERY = 50_000

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = "/content/hr-simulation"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")
!git log --oneline -3

In [ ]:
!pip install -q uv
!uv pip install --system -e "."

## Shared Utilities

In [ ]:
import glob
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv


class ColabLoggingCallback(BaseCallback):
    def __init__(self, log_every=10, total_timesteps=1_000_000, history_path=None):
        super().__init__(verbose=0)
        self._log_every = log_every
        self._total_timesteps = total_timesteps
        self._history_path = history_path
        self._ep_count = 0
        self._start = time.time()
        self.history = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self._ep_count += 1
                ep = info["episode"]
                self.history.append({
                    "episode": self._ep_count,
                    "timesteps": self.num_timesteps,
                    "reward": float(ep["r"]),
                    "length": int(ep["l"]),
                })
                if self._ep_count % self._log_every == 0:
                    elapsed = time.time() - self._start
                    ts = self.num_timesteps
                    rate = ts / elapsed if elapsed > 0 else 0
                    pct = ts / self._total_timesteps * 100
                    eta_s = (self._total_timesteps - ts) / rate if rate > 0 else 0
                    eta_m = eta_s / 60
                    recent = self.history[-10:]
                    avg_r = np.mean([h["reward"] for h in recent])
                    avg_l = np.mean([h["length"] for h in recent])
                    print(
                        f"  [{pct:5.1f}%] ep {self._ep_count:5d} | "
                        f"reward: {avg_r:8.3f} | "
                        f"len: {avg_l:6.0f} | "
                        f"ts: {ts:>10,} | "
                        f"{rate:,.0f} ts/s | "
                        f"ETA: {eta_m:.0f}m"
                    )
                if self._history_path and self._ep_count % (self._log_every * 5) == 0:
                    with open(self._history_path, "w") as f:
                        json.dump(self.history, f)
        return True


def find_checkpoint(drive_dir, total_timesteps):
    """Find latest checkpoint. Returns (path, restored_timesteps)."""
    if RESUME is False:
        return None, 0
    if RESUME != 'auto':
        return RESUME, 0

    zips = sorted(glob.glob(os.path.join(drive_dir, "checkpoint_*.zip")), key=os.path.getmtime)
    if zips:
        path = zips[-1]
        stem = Path(path).stem
        parts = stem.split("_")
        restored = 0
        for i, p in enumerate(parts):
            if p == "steps" and i > 0:
                try:
                    restored = int(parts[i - 1])
                except ValueError:
                    pass
        return path, restored

    final = os.path.join(drive_dir, "final_model.zip")
    if os.path.exists(final):
        return final, total_timesteps

    return None, 0


def plot_history(history, title="Training"):
    if not history:
        print("No history.")
        return
    rewards = [h["reward"] for h in history]
    lengths = [h["length"] for h in history]
    window = min(50, len(rewards) // 4) or 1
    smooth_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
    smooth_l = np.convolve(lengths, np.ones(window)/window, mode='valid')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title)
    ax1.plot(rewards, alpha=0.2, color='blue')
    ax1.plot(range(window-1, len(rewards)), smooth_r, color='blue', linewidth=2)
    ax1.set_xlabel('Episode'); ax1.set_ylabel('Reward'); ax1.set_title('Episode Reward')
    ax1.grid(True, alpha=0.3)
    ax2.plot(lengths, alpha=0.2, color='green')
    ax2.plot(range(window-1, len(lengths)), smooth_l, color='green', linewidth=2)
    ax2.set_xlabel('Episode'); ax2.set_ylabel('Length (ticks)'); ax2.set_title('Episode Length')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Episodes: {len(rewards)} | Last 10 avg reward: {np.mean(rewards[-10:]):.3f} | Last 10 avg length: {np.mean(lengths[-10:]):.0f}")


def run_training(vec_env, model, timesteps, drive_dir, restore_path):
    """Run training loop with checkpoints and logging."""
    history_path = os.path.join(drive_dir, "history.json")
    checkpoint_cb = CheckpointCallback(
        save_freq=max(SAVE_EVERY // N_ENVS, 1),
        save_path=drive_dir,
        name_prefix="checkpoint",
    )
    logging_cb = ColabLoggingCallback(
        log_every=LOG_EVERY,
        total_timesteps=timesteps,
        history_path=history_path,
    )
    start = time.time()
    try:
        model.learn(
            total_timesteps=timesteps,
            callback=[checkpoint_cb, logging_cb],
            reset_num_timesteps=False if restore_path else True,
        )
    except KeyboardInterrupt:
        print("\nTraining interrupted.")

    final_path = os.path.join(drive_dir, "final_model")
    model.save(final_path)
    with open(history_path, "w") as f:
        json.dump(logging_cb.history, f)

    elapsed = time.time() - start
    print(f"\n{'='*60}")
    print(f"Done — {elapsed/60:.1f} min, {logging_cb._ep_count} episodes")
    print(f"Final model: {final_path}.zip")
    print(f"{'='*60}")
    return logging_cb.history

---
## Phase 1: Solo Time Trial (learn pacing)

One horse, no opponents. Reward = `20 * delta_progress - 0.005/tick - 0.01/tick@exhausted + 10.0 on finish`.

Expected rewards: fast finish ~+22.5, cruise ~+19.5, push-hard-exhaust ~-12.5.

In [ ]:
from horse_racing.env.solo_env import SoloTimeTrialEnv

os.makedirs(SOLO_DIR, exist_ok=True)

restore_path, restored_ts = find_checkpoint(SOLO_DIR, SOLO_TIMESTEPS)
remaining = max(0, SOLO_TIMESTEPS - restored_ts)

print(f"Phase 1: Solo pacing")
print(f"  Dir: {SOLO_DIR}")
print(f"  Resume: {restore_path or 'fresh'}")
print(f"  Remaining: {remaining:,} timesteps")

In [ ]:
if remaining <= 0:
    print("Phase 1 already complete.")
    solo_model_path = os.path.join(SOLO_DIR, "final_model.zip")
else:
    def make_solo_env():
        def _init():
            return Monitor(SoloTimeTrialEnv(track_path=TRACK, max_steps=MAX_STEPS))
        return _init

    solo_vec = SubprocVecEnv([make_solo_env() for _ in range(N_ENVS)])
    policy_kwargs = dict(net_arch=[256, 256])

    if restore_path:
        print(f"Resuming from {restore_path}")
        solo_model = PPO.load(restore_path, env=solo_vec, device="cpu")
    else:
        solo_model = PPO(
            "MlpPolicy", solo_vec,
            learning_rate=3e-4,
            gamma=0.998,
            gae_lambda=0.95,
            n_steps=2048,
            batch_size=512,
            n_epochs=10,
            clip_range=0.2,
            ent_coef=0.02,
            policy_kwargs=policy_kwargs,
            verbose=0,
            device="cpu",
        )

    print(f"Params: {sum(p.numel() for p in solo_model.policy.parameters()):,}")
    solo_history = run_training(solo_vec, solo_model, remaining, SOLO_DIR, restore_path)
    solo_vec.close()
    solo_model_path = os.path.join(SOLO_DIR, "final_model.zip")

In [ ]:
# Phase 1 curves
hp = os.path.join(SOLO_DIR, "history.json")
if 'solo_history' in dir():
    h = solo_history
elif os.path.exists(hp):
    with open(hp) as f: h = json.load(f)
else:
    h = []
plot_history(h, "Phase 1: Solo Pacing")

In [ ]:
# Quick solo evaluation
from horse_racing.action import decode_action

eval_model = PPO.load(solo_model_path, device="cpu")
env = SoloTimeTrialEnv(track_path=TRACK, max_steps=MAX_STEPS)

print("Solo evaluation (5 episodes):")
for ep in range(5):
    obs, info = env.reset()
    total_r = 0.0
    steps = 0
    while True:
        action, _ = eval_model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = env.step(int(action))
        total_r += r
        steps += 1
        if term or trunc:
            break
    print(f"  ep {ep+1}: reward={total_r:8.3f} | steps={steps:5d} | progress={info['progress']:.3f} | stamina={info['stamina']:.1f}")
env.close()

---
## Phase 2: Race (transfer pacing → competitive)

Load the solo-trained policy into the race env with scripted opponents.
Lower learning rate to fine-tune without destroying pacing knowledge.

In [ ]:
from horse_racing.env import HorseRacingSingleEnv

os.makedirs(RACE_DIR, exist_ok=True)

# Check if phase 2 has its own checkpoints first
race_restore, race_restored_ts = find_checkpoint(RACE_DIR, RACE_TIMESTEPS)
race_remaining = max(0, RACE_TIMESTEPS - race_restored_ts)

print(f"Phase 2: Racing")
print(f"  Dir: {RACE_DIR}")
print(f"  Resume: {race_restore or 'from solo model'}")
print(f"  Remaining: {race_remaining:,} timesteps")

In [ ]:
if race_remaining <= 0:
    print("Phase 2 already complete.")
else:
    def make_race_env():
        def _init():
            return Monitor(HorseRacingSingleEnv(
                track_path=TRACK, horse_count=HORSE_COUNT, max_steps=MAX_STEPS,
            ))
        return _init

    race_vec = SubprocVecEnv([make_race_env() for _ in range(N_ENVS)])

    if race_restore:
        # Resume phase 2 from its own checkpoint
        print(f"Resuming phase 2 from {race_restore}")
        race_model = PPO.load(race_restore, env=race_vec, device="cpu")
    else:
        # Transfer from solo model
        print(f"Transferring from solo model: {solo_model_path}")
        race_model = PPO.load(solo_model_path, env=race_vec, device="cpu")

    # Use lower LR for fine-tuning
    race_model.learning_rate = RACE_LR
    race_model._setup_lr_schedule()
    print(f"  LR: {RACE_LR}")
    print(f"  Params: {sum(p.numel() for p in race_model.policy.parameters()):,}")

    race_history = run_training(race_vec, race_model, race_remaining, RACE_DIR, race_restore)
    race_vec.close()

In [ ]:
# Phase 2 curves
hp = os.path.join(RACE_DIR, "history.json")
if 'race_history' in dir():
    h = race_history
elif os.path.exists(hp):
    with open(hp) as f: h = json.load(f)
else:
    h = []
plot_history(h, "Phase 2: Racing")

## Evaluation

In [ ]:
# Evaluate the race-trained model
final_path = os.path.join(RACE_DIR, "final_model.zip")
if not os.path.exists(final_path):
    zips = sorted(glob.glob(os.path.join(RACE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    final_path = zips[-1] if zips else os.path.join(SOLO_DIR, "final_model.zip")

eval_model = PPO.load(final_path, device="cpu")
print(f"Evaluating: {final_path}\n")

env = HorseRacingSingleEnv(track_path=TRACK, horse_count=HORSE_COUNT, max_steps=MAX_STEPS)
for ep in range(5):
    obs, info = env.reset()
    total_r = 0.0
    steps = 0
    while True:
        action, _ = eval_model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = env.step(int(action))
        total_r += r
        steps += 1
        if term or trunc:
            break
    status = f"#{info['finish_order']}" if info['finish_order'] else "DNF"
    print(f"  ep {ep+1}: reward={total_r:8.3f} | steps={steps:5d} | progress={info['progress']:.3f} | stamina={info['stamina']:.1f} | {status}")
env.close()

## Export to ONNX

In [ ]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
from horse_racing.core.observation import OBS_SIZE
from horse_racing.action import NUM_ACTIONS


class PolicyWrapper(nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.features_extractor = policy.features_extractor
        self.mlp_extractor = policy.mlp_extractor
        self.action_net = policy.action_net

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        features = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(features)
        return self.action_net(latent_pi)


final_path = os.path.join(RACE_DIR, "final_model.zip")
if not os.path.exists(final_path):
    zips = sorted(glob.glob(os.path.join(RACE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    final_path = zips[-1] if zips else os.path.join(SOLO_DIR, "final_model.zip")

export_model = PPO.load(final_path, device="cpu")
wrapper = PolicyWrapper(export_model.policy).cpu()
wrapper.eval()

dummy = torch.zeros(1, OBS_SIZE, dtype=torch.float32)
onnx_path = os.path.join(RACE_DIR, f"{VERSION}_jockey.onnx")

torch.onnx.export(
    wrapper, dummy, onnx_path,
    input_names=["obs"], output_names=["actions"],
    dynamic_axes={"obs": {0: "batch"}, "actions": {0: "batch"}},
    opset_version=17,
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(onnx_path)
test_obs = np.random.randn(4, OBS_SIZE).astype(np.float32)
with torch.no_grad():
    torch_out = wrapper(torch.from_numpy(test_obs)).numpy()
onnx_out = session.run(["actions"], {"obs": test_obs})[0]
max_diff = np.max(np.abs(torch_out - onnx_out))

print(f"Exported to: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")
print(f"Max PyTorch vs ONNX diff: {max_diff:.2e}")
assert max_diff < 1e-5
print("Validation passed!")
print(f"\nDrive path: hr-checkpoints/{VERSION}-race/{VERSION}_jockey.onnx")
print(f"Copy to: apps/horse-racing/public/models/{VERSION}_jockey.onnx")